# Ćwiczenie 7 — Współczynnik uwarunkowania

Współczynnik uwarunkowania macierzy $A$ (oznaczany jako $\mathrm{cond}(A)$ lub $\kappa(A)$)
mierzy czułość rozwiązania układu $Ax=b$ na zaburzenia danych. Dla macierzy nieosobliwej:

$$\mathrm{cond}(A) = \|A\| \cdot \|A^{-1}\|$$

Jeśli $\mathrm{cond}(A) \approx 1$, macierz jest dobrze uwarunkowana.
Duża wartość współczynnika oznacza złą uwarunkowanie — błędy w danych mogą być mocno amplificowane.

W celu oceny numerycznej stabilności rozwiązań układów równań liniowych,
w ćwiczeniach poniżej porównujemy błędy rzeczywiste z oszacowaniami opartymi na współczynniku uwarunkowania.

In [ ]:
import numpy as np
import scipy.linalg
import matplotlib.pyplot as plt
from scipy.linalg import lu_factor, lu_solve, qr

def hilb(n):
    """Generuje macierz Hilberta n x n."""
    return np.array([[1.0/(i+j+1) for j in range(n)] for i in range(n)], dtype=np.float64)


def solveUsingQRFactorization(M, rhs):
    """Rozwiązanie układu równań liniowych MX=rhs przy użyciu dekompozycji QR."""
    Q, R = qr(M.astype(np.float64), mode="economic")
    Qt_rhs = Q.T @ rhs.astype(np.float64)
    x = scipy.linalg.solve_triangular(R, Qt_rhs)
    return x.astype(np.float64)

def solveUsingLUFactorization(M, rhs):
    """Rozwiązanie układu równań liniowych MX=rhs przy użyciu dekompozycji LU z pivotowaniem."""
    lu, piv = lu_factor(M.astype(np.float64))
    x = lu_solve((lu, piv), rhs.astype(np.float64))
    return x.astype(np.float64)

## ĆWICZENIE 1 — Porównanie błędów dla macierzy dobrze i źle uwarunkowanych

Porównujemy dwie macierze:

1. **Macierz dobrze uwarunkowana**: $A_1 = \text{magic}(i) + 10\cdot I$ — ma niewielki współczynnik uwarunkowania.
2. **Macierz źle uwarunkowana**: $A_2 = H_i$ (macierz Hilberta) — jej współczynnik rośnie wykładniczo z $i$.

Wektor prawych stron: $b = \sum_j A_{ij}$, czyli rzędów sumowane (rozwiązanie dokładne $x = [1,1,\dots,1]^T$).

Dla każdego $N$ obliczamy błąd rozwiązań oraz wartość $\mathrm{cond}(A)$ i przedstawiamy je na trzech wykresach.

In [ ]:
Nmax = 300
step = 20
Nvalues = list(range(4, Nmax + 1, step))

err_qr_1 = []
err_lu_1 = []
err_qr_2 = []
err_lu_2 = []
cond_1  = []
cond_2  = []

for i, N in enumerate(Nvalues):
    if (i + 1) % 5 == 0:
        print(f"Waitbar: {i+1}/{len(Nvalues)}")

    rng = np.random.RandomState(N * 137)
    A1 = (rng.rand(N, N) + 10 * np.eye(N)).astype(np.float64)
    b1 = A1.sum(axis=1).astype(np.float64)

    A2 = hilb(N).astype(np.float64)
    b2 = A2.sum(axis=1).astype(np.float64)

    x_true = np.ones(N, dtype=np.float64)

    x_qr1 = solveUsingQRFactorization(A1, b1)
    x_lu1 = solveUsingLUFactorization(A1, b1)
    x_qr2 = solveUsingQRFactorization(A2, b2)
    x_lu2 = solveUsingLUFactorization(A2, b2)

    err_qr_1.append(np.linalg.norm(x_qr1 - x_true))
    err_lu_1.append(np.linalg.norm(x_lu1 - x_true))
    err_qr_2.append(np.linalg.norm(x_qr2 - x_true))
    err_lu_2.append(np.linalg.norm(x_lu2 - x_true))

    cond_1.append(np.linalg.cond(A1))
    cond_2.append(np.linalg.cond(A2))

fig, axes = plt.subplots(3, 1, figsize=(10, 12))

axes[0].semilogy(Nvalues, err_qr_1, "b-o", label="QR", markersize=4)
axes[0].semilogy(Nvalues, err_lu_1, "r-s", label="LU", markersize=4)
axes[0].set_xlabel("N")
axes[0].set_ylabel("Błąd ||x - x'||")
axes[0].set_title("Macierz dobrze uwarunkowana (magic + 10·I)")
axes[0].legend()
axes[0].grid(True, which="both", linestyle="--", alpha=0.5)

axes[1].semilogy(Nvalues, err_qr_2, "b-o", label="QR", markersize=4)
axes[1].semilogy(Nvalues, err_lu_2, "r-s", label="LU", markersize=4)
axes[1].set_xlabel("N")
axes[1].set_ylabel("Błąd ||x - x'||")
axes[1].set_title("Macierz źle uwarunkowana (Hilbert)")
axes[1].legend()
axes[1].grid(True, which="both", linestyle="--", alpha=0.5)

axes[2].semilogy(Nvalues, cond_1, 'g-d', label='cond(A_1) (magic+10I)', markersize=4)
axes[2].semilogy(Nvalues, cond_2, 'm-^', label='cond(A_2) (Hilbert)', markersize=4)
axes[2].set_xlabel('N')
axes[2].set_ylabel('cond(A)')
axes[2].set_title('Współczynnik uwarunkowania vs N')
axes[2].legend()
axes[2].grid(True, which='both', linestyle='--', alpha=0.5)

fig.tight_layout()
plt.show()

## ĆWICZENIE 2a — Zaburzenie wektora prawych stron $b$

Dla układu $Ax=b$ z zaburzoną prawą stroną $\tilde{b} = b + \Delta b$,
górne oszacowanie względnej zmiany rozwiązania wynosi:

$$\frac{\|\Delta x\|}{\|x\|} \leq \mathrm{cond}(A) \cdot \frac{\|\Delta b\|}{\|b\|}$$

Generujemy macierz losową $A \in \mathbb{R}^{i\times i}$, perturbujemy $b$ o $\pm 10^{-6}$,
i porównujemy rzeczywisty błąd z oszacowaniem.

---

**ĆWICZENIE 2b — Zaburzenie macierzy $A$**

Dla zaburzonej macierzy $\tilde{A} = A + \Delta A$ oszacowanie względnej zmiany rozwiązania:

$$\frac{\|\Delta x\|}{\|x\|} \leq \frac{\mathrm{cond}(A) \cdot \varepsilon}{1 - \mathrm{cond}(A) \cdot \varepsilon}, \quad \varepsilon = \frac{\|\Delta A\|}{\|A\|}$$

Gdzie $\varepsilon$ to względne zaburzenie macierzy.
Warunek $\mathrm{cond}(A) \cdot \varepsilon < 1$ musi być spełniony, aby oszacowanie było sensowne.

In [ ]:
# ĆWICZENIE 2a — Zaburzenie wektora b
Nmax = 200
step = 10
Nvalues = list(range(4, Nmax + 1, step))

rel_err_actual_b = []
rel_err_estimated_b = []

for i, N in enumerate(Nvalues):
    if (i + 1) % 10 == 0:
        print(f"Waitbar: {i+1}/{len(Nvalues)}")

    A = np.random.rand(N, N).astype(np.float64)
    b = A.sum(axis=1).astype(np.float64)

    x_true = np.ones(N, dtype=np.float64)
    x_orig = solveUsingLUFactorization(A, b)

    delta_b_rel = (np.random.rand(N) - 0.5) * 2e-6
    b_perturbed = b * (1.0 + delta_b_rel)
    x_perturbed = solveUsingLUFactorization(A, b_perturbed)

    actual = np.linalg.norm(x_perturbed - x_orig) / np.linalg.norm(x_orig)
    rel_err_actual_b.append(actual)

    delta_b = b_perturbed - b
    eps_b = np.linalg.norm(delta_b) / np.linalg.norm(b)
    estimated = np.linalg.cond(A) * eps_b
    rel_err_estimated_b.append(estimated)

fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(Nvalues, rel_err_estimated_b, "r--o", label="Oszacowanie: cond(A)·||Δb||/||b||", markersize=4)
ax.semilogy(Nvalues, rel_err_actual_b,   "b--s", label="Błąd rzeczywisty ||Δx||/||x||",     markersize=4)
ax.set_xlabel("N")
ax.set_ylabel("Względny błąd")
ax.set_title("Ćwiczenie 2a — Porównanie błędu rzeczywistego z oszacowaniem (zaburzenie b)")
ax.legend()
ax.grid(True, which="both", linestyle="--", alpha=0.5)
fig.tight_layout()
plt.show()

In [ ]:
# ĆWICZENIE 2b — Zaburzenie macierzy A
Nmax = 200
step = 10
Nvalues = list(range(4, Nmax + 1, step))

rel_err_actual_A = []
rel_err_estimated_A = []

for i, N in enumerate(Nvalues):
    if (i + 1) % 10 == 0:
        print(f"Waitbar: {i+1}/{len(Nvalues)}")

    A = hilb(N).astype(np.float64)
    b = A.sum(axis=1).astype(np.float64)

    x_true = np.ones(N, dtype=np.float64)
    x_orig = solveUsingLUFactorization(A, b)

    delta_A = (np.random.rand(N, N) - 0.5) * 2e-6
    A_perturbed = A + delta_A
    x_perturbed = solveUsingLUFactorization(A_perturbed, b)

    actual = np.linalg.norm(x_perturbed - x_orig) / np.linalg.norm(x_orig)
    rel_err_actual_A.append(actual)

    eps_A = np.linalg.norm(delta_A) / np.linalg.norm(A)
    k = np.linalg.cond(A)
    denom = 1.0 - k * eps_A
    if denom > 0:
        estimated = abs(k * eps_A / denom)
    else:
        estimated = np.inf
    rel_err_estimated_A.append(estimated)

fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(Nvalues, rel_err_estimated_A, "r--o", label="Oszacowanie: |cond·ε/(1-cond·ε)|", markersize=4)
ax.semilogy(Nvalues, rel_err_actual_A,   "b--s", label="Błąd rzeczywisty ||Δx||/||x||",    markersize=4)
ax.set_xlabel("N")
ax.set_ylabel("Względny błąd")
ax.set_title("Ćwiczenie 2b — Porównanie błędu rzeczywistego z oszacowaniem (zaburzenie A, Hilbert)")
ax.legend()
ax.grid(True, which="both", linestyle="--", alpha=0.5)
fig.tight_layout()
plt.show()

## ĆWICZENIE 3 — Uwagi końcowe

Dla macierzy kwadratowych dekompozycja LU jest zazwyczaj skuteczniejsza niż QR —
daje mniejsze błędy numeryczne przy tym samym współczynniku uwarunkowania.
Powodem jest to, że QR wprowadza dodatkową transformację ortogonalną, która
może nieznacznie zwiększać błędy zaokrągleń.

Jednak dekompozycja QR ma istotną zaletę: pozwala na rozwiązywanie układów
**prostokątnych** (systemy przereklamowane lub niedookreślone), gdzie LU nie ma zastosowania.
Dla systemów przereklamowanych $Ax=b$ z $A \in \mathbb{R}^{m\times n}$, $m>n$,
QR daje rozwiązanie najmniejszych kwadratów.

W MATLAB/Octave operator `\` (`mldivide`) automatycznie wybiera odpowiednią metodę
w zależności od struktury macierzy (sparse, symetryczna, prostokątna, itp.).

---

## Wykorzystane funkcje

| Funkcja | Opis |
|---------|------|
| `np.linalg.cond(A)` | Oblicza współczynnik uwarunkowania macierzy $A$ |
| `np.linalg.norm(x)` | Norma euklidesowa wektora (dla macierzy — norma spektralna/frobeniusza) |
| `scipy.linalg.lu_factor`, `lu_solve` | Dekompozycja LU z pivotowaniem i rozwiązywanie układu |
| `scipy.linalg.qr` | Dekompozycja QR |
| `scipy.linalg.solve_triangular` | Rozwiązywanie układu trójkątnego |
| `hilb(n)` | własna — generuje macierz Hilberta $H_{ij} = \frac{1}{i+j-1}$ |
| `np.random.rand(m, n)` | Macierz $m\times n$ z ciągłych liczb losowych z $U(0,1)$ |
| `np.random.RandomState(seed)` | Generator liczb losowych z ustalonym ziarnem |